# Лабораторная работа №5
## Аппарат сетей Петри: параметризованная версия

В этой версии выполняется моделирование задачи об обедающих философах
не для одного набора значений, а для нескольких параметров.

Рассматриваются разные значения:
- числа философов `N`;
- времени моделирования `tmax`.

Для каждого набора параметров строятся:
- классическая сеть Петри;
- сеть Петри с арбитром.

Далее выполняется стохастическая симуляция, сохраняются таблицы с результатами,
проверяется наличие deadlock и строятся графики.

In [ ]:
using DrWatson
@quickactivate "project"

## Подключение библиотек

In [ ]:
using CSV
using DataFrames
using Plots

include(srcdir("DiningPhilosophers.jl"))
using .DiningPhilosophers

## Наборы параметров

Здесь задаются значения параметров, для которых будет выполнено моделирование.

In [ ]:
philosopher_counts = [3, 5, 7]
time_values = [30.0, 50.0]

## Подготовка каталогов

Результаты параметризованных запусков удобно хранить отдельно.

In [ ]:
mkpath(datadir("params"))
mkpath(plotsdir("params"))

## Запуск моделирования для набора параметров

Для каждого сочетания `N` и `tmax` выполняются:
- симуляция классической сети;
- симуляция сети с арбитром;
- сохранение CSV;
- проверка deadlock;
- сохранение графиков;
- добавление краткой сводки в итоговую таблицу.

In [ ]:
summary = DataFrame(
    N = Int[],
    tmax = Float64[],
    model = String[],
    deadlock = Bool[],
    n_records = Int[]
)

for N in philosopher_counts
    for tmax in time_values
        println("="^60)
        println("Параметры: N = $N, tmax = $tmax")

### Классическая сеть Петри

In [ ]:
        println("Запуск классической сети Петри...")
        net_classic, u0_classic, _ = build_classical_network(N)

        df_classic = simulate_stochastic(net_classic, u0_classic, tmax)

        classic_csv = datadir("params", "classic_N$(N)_t$(Int(round(tmax))).csv")
        CSV.write(classic_csv, df_classic)

        deadlock_classic = detect_deadlock(df_classic, net_classic)
        println("Deadlock обнаружен (классическая сеть): ", deadlock_classic)

        p_classic = plot_marking_evolution(df_classic, N)
        classic_plot = plotsdir("params", "classic_N$(N)_t$(Int(round(tmax))).png")
        savefig(p_classic, classic_plot)

        push!(summary, (
            N,
            tmax,
            "classic",
            deadlock_classic,
            nrow(df_classic)
        ))

### Сеть Петри с арбитром

In [ ]:
        println("Запуск сети Петри с арбитром...")
        net_arbiter, u0_arbiter, _ = build_arbiter_network(N)

        df_arbiter = simulate_stochastic(net_arbiter, u0_arbiter, tmax)

        arbiter_csv = datadir("params", "arbiter_N$(N)_t$(Int(round(tmax))).csv")
        CSV.write(arbiter_csv, df_arbiter)

        deadlock_arbiter = detect_deadlock(df_arbiter, net_arbiter)
        println("Deadlock обнаружен (сеть с арбитром): ", deadlock_arbiter)

        p_arbiter = plot_marking_evolution(df_arbiter, N)
        arbiter_plot = plotsdir("params", "arbiter_N$(N)_t$(Int(round(tmax))).png")
        savefig(p_arbiter, arbiter_plot)

        push!(summary, (
            N,
            tmax,
            "arbiter",
            deadlock_arbiter,
            nrow(df_arbiter)
        ))
    end
end

## Сохранение итоговой таблицы

В итоговой таблице фиксируется:
- число философов;
- время моделирования;
- тип модели;
- наличие deadlock;
- число сохранённых состояний.

In [ ]:
summary_file = datadir("params", "summary_dining_philosophers.csv")
CSV.write(summary_file, summary)

println("="^60)
println("Итоговая таблица:")
println(summary)

## Итог

После выполнения параметризованной версии:
- в `data/params/` сохраняются CSV-файлы по каждому запуску;
- в `plots/params/` сохраняются соответствующие графики;
- в `data/params/summary_dining_philosophers.csv` сохраняется сводная таблица.

Это позволяет сравнить поведение классической сети и сети с арбитром
для разных параметров моделирования.

In [ ]:
println("Готово.")
println("Параметризованные CSV сохранены в папке data/params/")
println("Параметризованные графики сохранены в папке plots/params/")
println("Сводка сохранена в файле: ", summary_file)